In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import *
from snowflake.snowpark.functions import *

session = get_active_session()
dfj = session.read.json("@exercise_db.public.stage1/car-sales.ndjson")
dfj

In [ ]:
df = dfj.select(sql_expr("$1:customer[0].name::string"))
df

In [ ]:
USE exercise_db.public;

SELECT src:dealership::string,
    src:salesperson.name::string,
    src['salesperson']['name'],
    src:customer[0].name,
    src:vehicle[0],
    src:vehicle[0].price::int,
    GET(src:vehicle[0], 'make'),
    GET_PATH(src:vehicle, '[0]')
FROM car_sales
ORDER BY 1;

In [ ]:
dft = session.table("exercise_db.public.car_sales")
df = dft.select(
    col("src")["dealership"].cast(StringType()),
    col("src")["salesperson"]["name"].cast(StringType()),
    col("src")['salesperson']['name'],
    col("src")["customer"][0]["name"],
    sql_expr("src:customer[0].name::string"),
    col("src")["vehicle"][0],
    col("src")["vehicle"][0]["price"].cast(IntegerType()),
    get(col("src")["vehicle"][0], lit("make")),
    get_path(col("src"), lit("vehicle[0]"))
)
df

In [ ]:
SELECT *
FROM car_sales, LATERAL FLATTEN(src:vehicle[0]);

In [ ]:
df = dft.join_table_function("flatten", col("src")["vehicle"][0])
df

In [ ]:
SELECT value, value:make, value:extras
FROM car_sales, LATERAL FLATTEN(src:vehicle);

In [ ]:
df = dft.join_table_function("flatten", col("src")["vehicle"]
    ).select(col("value"), col("value")["make"], col("value")["extras"])
df

In [ ]:
SELECT v.value as vehicle, e.value as extras
FROM car_sales,
    LATERAL FLATTEN(src:vehicle) v,
    LATERAL FLATTEN(v.value:extras) e;

In [ ]:
df = dft.join_table_function("flatten", col("src")["vehicle"]
    ).select(col("value").as_("vehicle"))
df = df.join_table_function("flatten", col("vehicle")["extras"]
    ).select(col("vehicle"), col("value").as_("extras"))
df